# Trabalho Prático 2 — Ciência de Dados
## Inflação sobre bens de consumo: Brasil e o mundo

### Objetivo

Este notebook inicia uma análise da evolução da inflação ao consumidor, com foco na comparação do Brasil com outros países e regiões do mundo.

O indicador utilizado é **“Inflation, consumer prices (annual %)” (FP.CPI.TOTL.ZG)**, disponibilizado pelo World Bank por meio dos World Development Indicators. A base enviada foi atualizada em **13/07/2026** e contém observações anuais de 1960 a 2025.

> **Observação metodológica:** o indicador mede a variação percentual anual dos preços ao consumidor. Portanto, ele representa uma **taxa de inflação**, e não o nível absoluto de preços nem o custo de uma cesta específica de bens.

## 1. Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import StringIO
from IPython.display import display

## 2. Carregamento dos dados

O CSV do World Bank possui linhas introdutórias antes do cabeçalho. Por isso, neste arquivo, `skiprows=4` é utilizado no dataset principal. O metadata possui cabeçalho tabular próprio e não deve receber o mesmo tratamento.

O carregamento é feito passando o conteúdo do arquivo por `StringIO`, mantendo a abordagem utilizada no trabalho anterior.

In [ ]:
with open("./API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_368812.csv", "r", encoding="utf-8-sig") as file:
    inflation_csv = file.read()

with open("./Metadata_Country_API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_368812.csv", "r", encoding="utf-8-sig") as file:
    metadata_csv = file.read()

inflation_df = pd.read_csv(StringIO(inflation_csv), skiprows=4)
country_meta = pd.read_csv(StringIO(metadata_csv))

print("Dataset principal:", inflation_df.shape)
print("Metadata:", country_meta.shape)

## 4. Estrutura inicial

O dataset principal apresenta os anos como colunas, isto é, está no formato **wide**. As colunas de identificação são `Country Name`, `Country Code`, `Indicator Name` e `Indicator Code`.

Os anos de 1960 a 2025 serão identificados automaticamente para evitar depender de uma lista digitada manualmente.

In [ ]:
year_columns = [col for col in inflation_df.columns if str(col).isdigit()]

print("Primeiro ano:", year_columns[0])
print("Último ano:", year_columns[-1])
print("Quantidade de anos:", len(year_columns))

display(inflation_df.head())

## 5. Normalização dos tipos

As colunas anuais serão convertidas para valores numéricos. Valores vazios do CSV são transformados em `NaN`, permitindo que o Pandas os reconheça como dados ausentes.

Não será feita imputação nesta etapa. Primeiro é necessário medir a cobertura dos dados para decidir quais anos e quais entidades podem sustentar a análise.

In [ ]:
inflation_df[year_columns] = inflation_df[year_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

print("Quantidade total de valores ausentes:")
print(inflation_df[year_columns].isna().sum().sum())

display(inflation_df[["Country Name", "Country Code"] + year_columns[:5]].head())

## 6. Integração com os metadados

A classificação será obtida pela coluna `Region` do metadata. Entidades com `Region` preenchida são países; entidades sem região são agregados do Banco Mundial.

Isso é particularmente importante neste estudo porque a base contém tanto países quanto agregados como **Arab World**, **High income**, **Low income** e agregados regionais. Não devemos misturá-los ao calcular médias entre países.

O código utiliza `Country Code` como chave, evitando problemas causados por diferenças de grafia nos nomes.

In [ ]:
country_meta["Type"] = np.where(
    country_meta["Region"].notna(),
    "Country",
    "Aggregated"
)

metadata_columns = [
    "Country Code",
    "Region",
    "IncomeGroup",
    "Type"
]

inflation_df = inflation_df.merge(
    country_meta[metadata_columns],
    on="Country Code",
    how="left"
)

print(inflation_df["Type"].value_counts(dropna=False))
display(inflation_df[[
    "Country Name", "Country Code", "Region", "IncomeGroup", "Type"
]].head(10))

## 7. Verificação de inconsistências

Para este indicador, valores negativos **não são necessariamente inconsistências**: uma taxa negativa representa deflação. Portanto, diferentemente da taxa de conclusão educacional do trabalho anterior, não devemos substituir automaticamente valores menores que zero por `NaN`.

Nesta etapa, o foco será detectar valores não numéricos e verificar a existência de valores extremos. Valores muito altos de inflação podem representar episódios econômicos reais e, por isso, não devem ser removidos apenas por serem grandes.

In [ ]:
invalid_count = inflation_df[year_columns].isna().sum().sum()

print("Valores ausentes:", invalid_count)
print("Menor taxa observada:", inflation_df[year_columns].min().min())
print("Maior taxa observada:", inflation_df[year_columns].max().max())

display(
    inflation_df[year_columns]
    .stack()
    .describe()
)

## 8. Separação entre países e agregados

Serão mantidas duas visões da base:

- `inflation_countries`: somente países, destinada às comparações entre países, rankings e cálculo de estatísticas por região.
- `inflation_aggregates`: agregados fornecidos pelo Banco Mundial, que podem ser úteis para comparação e validação, mas não serão misturados às estatísticas calculadas a partir dos países.

A base original `inflation_df` continua preservada.

In [ ]:
inflation_countries = inflation_df[
    inflation_df["Type"] == "Country"
].copy()

inflation_aggregates = inflation_df[
    inflation_df["Type"] == "Aggregated"
].copy()

print("Países:", len(inflation_countries))
print("Agregados:", len(inflation_aggregates))

## 9. Cobertura dos dados por ano

Antes de escolher o período de análise, será calculada a proporção de países com valores válidos em cada ano.

Esse diagnóstico permitirá definir um critério objetivo para o período analisado, em vez de escolher arbitrariamente um ano inicial. A imputação de valores faltantes será tratada somente depois dessa decisão.

In [ ]:
coverage = inflation_countries[year_columns].notna().mean()

coverage_df = pd.DataFrame({
    "Year": coverage.index.astype(int),
    "Coverage": coverage.values,
    "Coverage (%)": coverage.values * 100
})

display(coverage_df.head())
display(coverage_df.tail())

## 10. Visualização inicial da cobertura

O gráfico abaixo permite identificar visualmente em quais períodos a base apresenta maior disponibilidade de dados. A partir dele poderá ser estabelecido, na próxima etapa, um limiar mínimo de cobertura para a análise principal.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(coverage_df["Year"], coverage_df["Coverage (%)"])
plt.xlabel("Ano")
plt.ylabel("Países com dados válidos (%)")
plt.title("Cobertura dos dados de inflação por ano")
plt.grid(True, alpha=0.3)
plt.show()

## 11. Próximas etapas

A partir desta base normalizada, as próximas etapas recomendadas são:

1. Definir o período de análise com base na cobertura dos países.
2. Transformar a base de **wide para long** com `melt()`.
3. Avaliar a necessidade de interpolação, sem substituir valores negativos legítimos.
4. Comparar o Brasil com:
   - média dos países;
   - regiões do Banco Mundial;
   - grupos de renda;
   - países selecionados.
5. Construir séries temporais e indicadores específicos para o Brasil.
6. Investigar episódios de inflação elevada e sua duração.
7. Estruturar as visualizações finais para o dashboard.

**Ponto metodológico importante:** a média simples dos países não deve ser apresentada automaticamente como “inflação mundial”. Uma média entre países dá o mesmo peso a cada país. Caso o dashboard queira representar a inflação global ponderada pela participação econômica de cada país, será necessário incorporar outro indicador ou variável de ponderação.